In [32]:
!pip install transformers -q


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import math
from torch.utils.data import DataLoader, Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

import tensorflow as tf
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = \
    tf.keras.datasets.imdb.load_data(num_words=10000)

word_index = tf.keras.datasets.imdb.get_word_index()
index_word = {v+3: k for k, v in word_index.items()}
index_word[0] = '[PAD]'
index_word[1] = '[START]'
index_word[2] = '[UNK]'

def decode_review(sequence):
    return ' '.join([index_word.get(i, '?') for i in sequence])

print("Decoding reviews...")
X_train_text = [decode_review(x) for x in X_train_raw]
X_test_text  = [decode_review(x) for x in X_test_raw]

print(f"Sample review: {X_train_text[0][:100]}...")
print(f"Label        : {y_train_raw[0]}")

Device: cuda
Decoding reviews...
Sample review: [START] this film was just brilliant casting location scenery story direction everyone's really suit...
Label        : 1


In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification

print(f"Device: {device}")

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

sample_text = "The movie was absolutely fantastic!"
tokens = tokenizer(sample_text, 
                   padding='max_length',
                   max_length=20,
                   truncation=True,
                   return_tensors='pt')

print(f"Original text    : {sample_text}")
print(f"Input IDs        : {tokens['input_ids']}")
print(f"Attention mask   : {tokens['attention_mask']}")
print(f"Decoded back     : {tokenizer.decode(tokens['input_ids'][0])}")
print(f"\nSpecial tokens:")
print(f"[CLS] token id   : {tokenizer.cls_token_id}")
print(f"[SEP] token id   : {tokenizer.sep_token_id}")
print(f"[PAD] token id   : {tokenizer.pad_token_id}")

Device: cuda
Original text    : The movie was absolutely fantastic!
Input IDs        : tensor([[  101,  1996,  3185,  2001,  7078, 10392,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0]])
Attention mask   : tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
Decoded back     : [CLS] the movie was absolutely fantastic! [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]

Special tokens:
[CLS] token id   : 101
[SEP] token id   : 102
[PAD] token id   : 0


In [ ]:
MAX_LEN = 256  

class BERTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length     = self.max_len,
            padding        = 'max_length',
            truncation     = True,
            return_tensors = 'pt'
        )
        return {
            'input_ids'     : encoding['input_ids'].squeeze(),  
            'attention_mask': encoding['attention_mask'].squeeze(),  
            'label'         : torch.tensor(self.labels[idx], dtype=torch.float)
        }

train_dataset = BERTDataset(X_train_text, y_train_raw, tokenizer, MAX_LEN)
test_dataset  = BERTDataset(X_test_text,  y_test_raw,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches : {len(test_loader)}")

batch = next(iter(train_loader))
print(f"\nInput IDs shape     : {batch['input_ids'].shape}")
print(f"Attention mask shape: {batch['attention_mask'].shape}")
print(f"Labels shape        : {batch['label'].shape}")

Train batches: 1563
Test batches : 1563

Input IDs shape     : torch.Size([16, 256])
Attention mask shape: torch.Size([16, 256])
Labels shape        : torch.Size([16])


In [36]:
from transformers import BertForSequenceClassification


model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels = 1 
)

model = model.to(device)

total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters    : 109,483,009
Trainable parameters: 109,483,009


In [37]:
from transformers import get_linear_schedule_with_warmup


optimizer = torch.optim.AdamW(model.parameters(), 
                               lr=2e-5,       
                               weight_decay=0.01)

n_epochs     = 3 
total_steps  = len(train_loader) * n_epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = total_steps // 10, 
    num_training_steps = total_steps
)

criterion = nn.BCEWithLogitsLoss()

print(f"Total training steps : {total_steps}")
print(f"Warmup steps         : {total_steps // 10}")
print(f"Learning rate        : 2e-5")
print(f"Epochs               : {n_epochs}")

Total training steps : 4689
Warmup steps         : 468
Learning rate        : 2e-5
Epochs               : 3


In [38]:
def train_epoch_bert(model, loader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    correct    = 0
    total      = 0
    
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)
        
        optimizer.zero_grad()
        
        outputs = model(input_ids=input_ids, 
                       attention_mask=attention_mask)
        
        logits = outputs.logits.squeeze()
        loss   = criterion(logits, labels)
        
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()  
        
        total_loss += loss.item()
        predicted   = (torch.sigmoid(logits) >= 0.5).float()
        correct    += (predicted == labels).sum().item()
        total      += labels.size(0)
    
    return total_loss / len(loader), 100 * correct / total


def evaluate_bert(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct    = 0
    total      = 0
    
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)
            
            outputs = model(input_ids=input_ids,
                           attention_mask=attention_mask)
            logits  = outputs.logits.squeeze()
            loss    = criterion(logits, labels)
            
            total_loss += loss.item()
            predicted   = (torch.sigmoid(logits) >= 0.5).float()
            correct    += (predicted == labels).sum().item()
            total      += labels.size(0)
    
    return total_loss / len(loader), 100 * correct / total


best_val_acc     = 0.0
patience         = 2
patience_counter = 0

print("Fine-tuning BERT...")
print("="*50)

for epoch in range(n_epochs):
    train_loss, train_acc = train_epoch_bert(model, train_loader,
                                              criterion, optimizer,
                                              scheduler, device)
    val_loss, val_acc     = evaluate_bert(model, test_loader,
                                           criterion, device)
    
    if val_acc > best_val_acc:
        best_val_acc     = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), 'best_bert.pth')
    else:
        patience_counter += 1
    
    print(f"Epoch {epoch+1}/{n_epochs} | "
          f"Train: {train_acc:.1f}% | "
          f"Val: {val_acc:.1f}% | "
          f"Patience: {patience_counter}/{patience}")
    
    if patience_counter >= patience:
        print("Early stopping")
        break

print(f"\nBest Val Accuracy: {best_val_acc:.2f}%")

print(f"\n{'='*45}")
print(f"FINAL COMPARISON")
print(f"{'='*45}")
print(f"Vanilla RNN  : 76.92%")
print(f"LSTM         : 88.13%")
print(f"BiLSTM       : 88.48%")
print(f"Transformer  : 86.59%")
print(f"BERT         : {best_val_acc:.2f}%")

Fine-tuning BERT...
Epoch 1/3 | Train: 86.5% | Val: 91.9% | Patience: 0/2
Epoch 2/3 | Train: 94.5% | Val: 92.2% | Patience: 0/2
Epoch 3/3 | Train: 97.6% | Val: 92.6% | Patience: 0/2

Best Val Accuracy: 92.58%

FINAL COMPARISON
Vanilla RNN  : 76.92%
LSTM         : 88.13%
BiLSTM       : 88.48%
Transformer  : 86.59%
BERT         : 92.58%
